# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/usmanumer038/ml-internship-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### **Rule:** I will prioritize content that has enough search visibility but has a low CTR and/or has not been updated recently. These signals are useful because low CTR can indicate an opportunity to improve the search-result presentation, while stale content may need a refresh.

### Reason code: SEO_REFRESH

Action: REVIEW_AND_REFRESH

The rule uses current performance information only. It does not use future-window or label-derived information.

In [6]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df), "Columns:", len(df.columns))

print("\nCTR buckets:")
print(pd.cut(df["ctr"], [-1, .02, .05, .10, 1]).value_counts().sort_index())

print("\nStaleness buckets:")
print(pd.cut(df["days_since_last_update"], [-1, 30, 90, 180, np.inf]).value_counts().sort_index())

Rows: 30000 Columns: 44

CTR buckets:
ctr
(-1.0, 0.02]    13458
(0.02, 0.05]      961
(0.05, 0.1]      2077
(0.1, 1.0]      11815
Name: count, dtype: int64

Staleness buckets:
days_since_last_update
(-1.0, 30.0]     20480
(30.0, 90.0]       175
(90.0, 180.0]     9171
(180.0, inf]       174
Name: count, dtype: int64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Build the ranked queue (writes the CSV)
I score content using low CTR and content staleness. Higher scores receive higher review priority.

In [8]:
d = df.dropna(subset=["ctr","impressions_90d","days_since_last_update"])
d = d[d["impressions_90d"] >= 100].copy()

d["score"] = (
    .6 * d["ctr"].rank(pct=True, ascending=True) +
    .4 * d["days_since_last_update"].rank(pct=True)
)

d["reason_code"] = np.where(d["ctr"] < .05, "LOW_CTR", "STALE_CONTENT")
d["action"] = np.where(d["reason_code"] == "LOW_CTR",
                       "REVIEW_TITLE_META", "REFRESH_CONTENT")

d = d.sort_values("score", ascending=False)
d["rank"] = range(1, len(d) + 1)

out = d[["rank","content_id","score","reason_code","action",
         "ctr","impressions_90d","days_since_last_update"]]

os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)

display(out.head(20))

,rank,content_id,score,reason_code,action,ctr,impressions_90d,days_since_last_update
1735,1,content_c6a9f1c16dee,0.999773,STALE_CONTENT,REFRESH_CONTENT,11.67,180,211
6233,2,content_ba00ffc6318c,0.999364,STALE_CONTENT,REFRESH_CONTENT,4.93,345,211
191,3,content_4729b57ca036,0.998918,STALE_CONTENT,REFRESH_CONTENT,3.28,335,301
665,4,content_e444c00065bd,0.998914,STALE_CONTENT,REFRESH_CONTENT,3.38,148,211
8006,5,content_cb7e312f5d32,0.997405,STALE_CONTENT,REFRESH_CONTENT,2.45,21272,151
9158,6,content_a001aa4be7b7,0.996974,STALE_CONTENT,REFRESH_CONTENT,2.62,7485,106
12400,7,content_482aff19e9cc,0.996592,STALE_CONTENT,REFRESH_CONTENT,2.43,26287,106
40,8,content_4df0b7207fe3,0.993070,STALE_CONTENT,REFRESH_CONTENT,1.65,1700,151
16648,9,content_69fad7e6c50c,0.986545,STALE_CONTENT,REFRESH_CONTENT,1.32,28000,106
13070,10,content_c8f7e3b9891c,0.976184,STALE_CONTENT,REFRESH_CONTENT,1.01,12029,106


## 3. Top-20 review
The top 20 are reviewed as decision-support recommendations. A recommendation could be wrong if low CTR is expected for the search intent or if the content was recently improved.

In [9]:
d = df.dropna(subset=["ctr","impressions_90d","days_since_last_update"])
d = d[d["impressions_90d"] >= 100].copy()

d["score"] = (
    .6 * d["ctr"].rank(pct=True, ascending=True) +
    .4 * d["days_since_last_update"].rank(pct=True)
)

d["reason_code"] = np.where(d["ctr"] < .05, "LOW_CTR", "STALE_CONTENT")
d["action"] = np.where(d["reason_code"] == "LOW_CTR",
                       "REVIEW_TITLE_META", "REFRESH_CONTENT")

d = d.sort_values("score", ascending=False)
d["rank"] = range(1, len(d) + 1)

out = d[["rank","content_id","score","reason_code","action",
         "ctr","impressions_90d","days_since_last_update"]]

os.makedirs("work/outputs", exist_ok=True)
out.to_csv("work/outputs/baseline_action_score.csv", index=False)

print("CSV created successfully")
display(out.head(20))

CSV created successfully


,rank,content_id,score,reason_code,action,ctr,impressions_90d,days_since_last_update
1735,1,content_c6a9f1c16dee,0.999773,STALE_CONTENT,REFRESH_CONTENT,11.67,180,211
6233,2,content_ba00ffc6318c,0.999364,STALE_CONTENT,REFRESH_CONTENT,4.93,345,211
191,3,content_4729b57ca036,0.998918,STALE_CONTENT,REFRESH_CONTENT,3.28,335,301
665,4,content_e444c00065bd,0.998914,STALE_CONTENT,REFRESH_CONTENT,3.38,148,211
8006,5,content_cb7e312f5d32,0.997405,STALE_CONTENT,REFRESH_CONTENT,2.45,21272,151
9158,6,content_a001aa4be7b7,0.996974,STALE_CONTENT,REFRESH_CONTENT,2.62,7485,106
12400,7,content_482aff19e9cc,0.996592,STALE_CONTENT,REFRESH_CONTENT,2.43,26287,106
40,8,content_4df0b7207fe3,0.993070,STALE_CONTENT,REFRESH_CONTENT,1.65,1700,151
16648,9,content_69fad7e6c50c,0.986545,STALE_CONTENT,REFRESH_CONTENT,1.32,28000,106
13070,10,content_c8f7e3b9891c,0.976184,STALE_CONTENT,REFRESH_CONTENT,1.01,12029,106


## 4. Weak picks + leakage check

Some weak picks may appear because the baseline rule is simple. Low CTR or stale content does not guarantee that an action is needed. The recommendations are for decision-support only. No future-window, label-derived inputs, or product flags were used.

In [10]:
print("Weak picks:")
display(out[out["impressions_90d"] < 300].head(5))

print("\nLeakage check:")
print("Signals used: ctr, impressions_90d, days_since_last_update")
print("No future-window or label-derived inputs were used.")
print("No product flags were used.")

Weak picks:


,rank,content_id,score,reason_code,action,ctr,impressions_90d,days_since_last_update
1735,1,content_c6a9f1c16dee,0.999773,STALE_CONTENT,REFRESH_CONTENT,11.67,180,211
665,4,content_e444c00065bd,0.998914,STALE_CONTENT,REFRESH_CONTENT,3.38,148,211
18885,12,content_180fabd3a3e0,0.962928,STALE_CONTENT,REFRESH_CONTENT,0.83,120,183
18508,15,content_3c770ef0121a,0.954408,STALE_CONTENT,REFRESH_CONTENT,0.75,268,183
3042,19,content_0d3276c90637,0.925175,STALE_CONTENT,REFRESH_CONTENT,4.96,121,104



Leakage check:
Signals used: ctr, impressions_90d, days_since_last_update
No future-window or label-derived inputs were used.
No product flags were used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.